In [ ]:
import os
from pathlib import Path
import pickle
import sys
import subprocess

from tqdm import tqdm

import cv2
import matplotlib.pyplot as plt
import numpy as np
import IPython

In [ ]:
RANDOM_SEED = 13131313
np.random.seed(RANDOM_SEED)

ROOT_DIR = os.path.abspath('..')  # ROOT_DIR = Path(__file__).parents[1].resolve()
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)
RESEARCH_DIR = ROOT_DIR / Path('researches')

ORIGINAL_DATA_PATH = ROOT_DIR / Path('data/nesmdb24_seprsco/train')
RANDOM_SONG_PATH = ORIGINAL_DATA_PATH / Path('297_SkyKid_00_01StartMusicBGMIntroBGM.seprsco.pkl')

In [ ]:
# # TODO: To DP

# import subprocess
# ORIGINAL_DATA_DOWNLOAD_LINK = 'http://deepyeti.ucsd.edu/cdonahue/nesmdb/nesmdb24_seprsco.tar.gz'
# DEFAULT_DOWNLOAD_PATH = ROOT_DIR / Path('data')

# dest_str = str(DEFAULT_DOWNLOAD_PATH.resolve())

# output1 = subprocess.check_call(['wget', ORIGINAL_DATA_DOWNLOAD_LINK, '-P', dest_str])
# print(output1)
# output2 = subprocess.check_call(['tar', 'xvfz', f'{dest_str}/nesmdb24_seprsco.tar.gz', '-C', dest_str])
# print(output2)
# os.remove(DEFAULT_DOWNLOAD_PATH / 'nesmdb24_seprsco.tar.gz')

# SEPRSCO Format

In [ ]:
HIST_RANGE = 108

cnt, mins, mins_ex_zero, mean_sums, maxs = 0, np.array([255, 255, 255, 255], dtype='uint8'), \
    np.array([255, 255, 255, 255], dtype='uint8'), np.array([0.0, 0.0, 0.0, 0.0], dtype='float64'), \
    np.array([0, 0, 0, 0], dtype='uint8')
bins, hists = np.arange(HIST_RANGE + 1), [np.zeros([HIST_RANGE,]) for _ in range(4)]

for file in ORIGINAL_DATA_PATH.iterdir():
    with open(file, 'rb') as f:
        rate, nsamps, seprsco = pickle.load(f)

        mins = np.minimum(mins, seprsco.min(axis=0))
        mean_sums += seprsco.mean(axis=0)
        maxs = np.maximum(maxs, seprsco.max(axis=0))
        cnt += 1

        for hi in range(4):
            hists[hi] += np.histogram(seprsco[:, hi], bins=bins)[0]

        seprsco[seprsco == 0] = 255
        mins_ex_zero = np.minimum(mins_ex_zero, seprsco.min(axis=0))

print(f'Number of samples: {cnt}.')
print(f'Min values: {mins}.')
print(f'Non-zero min values: {mins_ex_zero}.')
print(f'Mean values: {mean_sums / cnt}.')
print(f'Max values: {maxs}.')

In [ ]:
plt.figure()

fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(12, 10))

x = np.arange(HIST_RANGE)
axs[0][0].bar(x, hists[0])
axs[0][1].bar(x, hists[1])
axs[1][0].bar(x, hists[2])
axs[1][1].bar(x, hists[3])

for i in range(2):
    for j in range(2):
        axs[i][j].grid()

**Instruments [[NESMDB README](https://github.com/chrisdonahue/nesmdb#nes-synthesizer-primer)]:**

- Pulse 1 (P1): {0, 32, ..., 108}

- Pulse 2 (P2): {0, 32, ..., 108}

- Triangle (TR): {0, 21, ..., 108}

- Noise (NO): {0, 1, ... 16}

**Why non-zero min values are 33, not 32?**

In [ ]:
subprocess.check_call(['python2', '-m', 'nesmdb.convert', 'seprsco_to_wav',
                       str(RANDOM_SONG_PATH.resolve()), '--out_dir', RESEARCH_DIR])
song_wav_path = RESEARCH_DIR / Path('.'.join([RANDOM_SONG_PATH.name[:-4], 'wav']))
IPython.display.Audio(str(song_wav_path.resolve()))

In [ ]:
os.remove(song_wav_path)

# Scaling - MinMax

In [ ]:
# Min values leave places for zeros
PULSES12_MIN_VAL = 31
PULSES12_MAX_VAL = 108
TRIANGLE_MIN_VAL = 20
TRIANGLE_MAX_VAL = 108
NOISE_MIN_VAL = 0
NOISE_MAX_VAL = 16


def read_seprsco_song(song_path: Path):
    with open(song_path, 'rb') as f:
        _, _, seprsco = pickle.load(f)
    return seprsco.T


def scale_instrument_min_max(inst_data: np.ndarray, inst_min: float, inst_max: float):
    inst_data[inst_data == 0] = inst_min
    inst_data -= inst_min
    inst_data /= (inst_max - inst_min)


def scale_song_min_max(song: np.ndarray) -> np.ndarray:
    song_copy = song.astype(np.float)

    scale_instrument_min_max(song_copy[:2, :], PULSES12_MIN_VAL, PULSES12_MAX_VAL)
    scale_instrument_min_max(song_copy[2, :], TRIANGLE_MIN_VAL, TRIANGLE_MAX_VAL)
    scale_instrument_min_max(song_copy[3, :], NOISE_MIN_VAL, NOISE_MAX_VAL)
    
    return song_copy


def unscale_instrument_min_max(inst_data: np.ndarray, inst_min: float, inst_max: float):
    inst_data *= (inst_max - inst_min)
    inst_data += inst_min
    inst_data[inst_data == inst_min] = 0


def unscale_song_min_max(song: np.ndarray) -> np.ndarray:
    song_copy = song.copy()
    
    unscale_instrument_min_max(song_copy[:2, :], PULSES12_MIN_VAL, PULSES12_MAX_VAL)
    unscale_instrument_min_max(song_copy[2, :], TRIANGLE_MIN_VAL, TRIANGLE_MAX_VAL)
    unscale_instrument_min_max(song_copy[3, :], NOISE_MIN_VAL, NOISE_MAX_VAL)
    
    return song_copy.astype(np.uint8)


def save_seprsco_song(song_data: np.ndarray, song_path: Path):
    with open(song_path, 'wb') as f:
        pickle.dump(song_data, f, protocol=2)

In [ ]:
song = read_seprsco_song(RANDOM_SONG_PATH)
print(song)
song = scale_song_min_max(song)
print(song)
song = unscale_song_min_max(song)
print(song)

# Representations

In [ ]:
FILES_CNT = 10

scaled_data, fi = [], 0
for file in ORIGINAL_DATA_PATH.iterdir():
    if fi >= FILES_CNT:
        break
    fi += 1
    scaled_data.append(scale_song_min_max(read_seprsco_song(file)))

len(scaled_data)

## Piano Roll
### Forward

In [ ]:
# for interpolation_method in [None, 'none', 'nearest', 'bilinear', 'bicubic', 'spline16',
#                              'spline36', 'hanning', 'hamming', 'hermite', 'kaiser', 'quadric',
#                              'catrom', 'gaussian', 'bessel', 'mitchell', 'sinc', 'lanczos']:
#     #for aspect in ['auto', 'equal']:
#     plt.matshow(song, interpolation=interpolation_method, aspect='auto')
#     plt.plot(f'{interpolation_method}')

for song in scaled_data:
    plt.matshow(song, interpolation='none', aspect='auto')

In [ ]:
SAMPLE_LEN = 64  # 128 = 4-5sec
ROWS_CNT = 4

def represent_as_piano_roll_stack(song: np.ndarray, rows: int):
    if song.shape[1] % rows != 0:
        raise ValueError(song.shape[1], rows)
    
    delta = song.shape[1] // rows
    stack_list = []
    for ri in range(1, rows + 1):
        stack_list.append(song[:, (ri-1) * delta : ri * delta])
        
    return np.stack(stack_list).reshape(4 * rows, delta)


def unrepresent_as_piano_roll_stack(song: np.ndarray, rows: int):
    stack_list = np.split(song, rows)
    return np.concatenate(stack_list, axis=1)

    
for song in scaled_data:
    if song.shape[1] < SAMPLE_LEN:
        raise ValueError(song.shape[1])
    song = represent_as_piano_roll_stack(song[:, :SAMPLE_LEN], ROWS_CNT)
    plt.matshow(song, interpolation='none', aspect='auto')

In [ ]:
# TODO: Move to DP
TRAIN_DATA_PATH = ROOT_DIR / Path('_'.join(['data/processed_train_data_rol', f'len{SAMPLE_LEN}',
                                            f'rows{ROWS_CNT}']))
img_path = TRAIN_DATA_PATH / Path('images')
bin_path = TRAIN_DATA_PATH / Path('binary')
os.makedirs(img_path, exist_ok=True)
os.makedirs(bin_path, exist_ok=True)


# TODO: Not all data! SAMPLES_NUM_FROM_SONG
for file in tqdm(ORIGINAL_DATA_PATH.iterdir()):
    song = read_seprsco_song(file)
    song = scale_song_min_max(song)
    
    if song.shape[1] < SAMPLE_LEN:
        continue
    song = represent_as_piano_roll_stack(song[:, -SAMPLE_LEN:], ROWS_CNT)
    
    plt.matshow(song, interpolation='none', aspect='auto')
    plt.savefig(str(img_path / Path('.'.join([file.name[:-4], 'jpg']))))
    np.save(str(bin_path / Path('.'.join([file.name[:-4], 'npy']))), song)

### Backward

In [ ]:
tmp_path = ROOT_DIR / Path('researches/tmp_binary.npy')

original_song = read_seprsco_song(RANDOM_SONG_PATH)[:, -SAMPLE_LEN:]

song = scale_song_min_max(original_song)
song = represent_as_piano_roll_stack(song, ROWS_CNT)
np.save(str(tmp_path), song)
print(song.shape)
        
read_song = np.load(tmp_path)
song = unrepresent_as_piano_roll_stack(song, ROWS_CNT)
song = unscale_song_min_max(song)

np.testing.assert_array_equal(original_song, song)

## To wav, to spectrogram

In [ ]:
subprocess.check_call(['python2', '-m', 'nesmdb.convert', 'seprsco_to_wav',
                       str(RANDOM_SONG_PATH.resolve()), '--out_dir', RESEARCH_DIR])
song_wav_path = RESEARCH_DIR / Path('.'.join([RANDOM_SONG_PATH.name[:-4], 'wav']))
IPython.display.Audio(str(song_wav_path.resolve()))

## MEL

In [ ]:
# TODO: Move results here. Bad approach for NES music.

### Фурье
https://www.kaggle.com/code/rftexas/converting-sounds-into-images-a-general-guide